# LeetCode 872 — Leaf-Similar Trees bằng Python

**Link đề:** https://leetcode.com/problems/leaf-similar-trees/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def leafSimilar(self, root1: TreeNode | None, root2: TreeNode | None) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu cây nhị phân (binary tree) ở mức node: `val`, `left`, `right`, `None`; biết thế nào là **lá** (leaf).
2. Nắm **DFS** (duyệt theo chiều sâu) bằng đệ quy và bằng stack, và vì sao DFS "trái trước phải sau" cho lá theo thứ tự trái → phải.
3. Biết cách biến một phép duyệt thành **dãy giá trị** (leaf sequence) rồi so sánh hai dãy.
4. Nâng cấp: dùng **generator** (`yield`) để so sánh từng lá và **dừng sớm** khi khác nhau.
5. Tự trace, chứng minh tính đúng, phân tích Time/Space và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Xét tất cả các **lá** của một cây nhị phân, đọc từ **trái sang phải**: giá trị của chúng tạo thành **dãy giá trị lá** (leaf value sequence).

Hai cây gọi là **leaf-similar** nếu dãy giá trị lá của chúng **giống hệt nhau**. Trả về `True` khi và chỉ khi `root1` và `root2` leaf-similar.

Ví dụ 1 (cây trong hình đề):

```text
            3                         3
         /     \                   /     \
        5       1                 5       1
       / \     / \               / \     / \
      6   2   9   8             6   7   4   2
         / \                               / \
        7   4                             9   8
```
- Lá của cây 1, trái → phải: `6, 7, 4, 9, 8`
- Lá của cây 2, trái → phải: `6, 7, 4, 9, 8`
- Giống nhau → **`true`**.

Ví dụ 2: `root1 = [1,2,3]`, `root2 = [1,3,2]` → lá là `(2,3)` và `(3,2)` → **`false`** (cùng tập giá trị nhưng **khác thứ tự**).

**Constraints:**
- Mỗi cây có từ `1` đến `200` node.
- Giá trị node trong `[0, 200]`.

Lưu ý: hình dạng cây **không cần** giống nhau, chỉ dãy lá cần giống nhau.

## 2. Nền tảng Python và cấu trúc dữ liệu cần biết

### 2.1. `TreeNode` — một node của cây nhị phân

Mỗi node có 3 thuộc tính:
- `val`: giá trị.
- `left`: con trái (một `TreeNode` khác hoặc `None`).
- `right`: con phải.

LeetCode tự cung cấp class này khi chấm; trong notebook ta tự định nghĩa để chạy độc lập.

In [1]:
from typing import Optional

class TreeNode:
    def __init__(self, val: int = 0, left: "Optional[TreeNode]" = None, right: "Optional[TreeNode]" = None):
        self.val = val
        self.left = left
        self.right = right

# Tạo tay cây nhỏ:   1
#                   / \
#                  2   3
root = TreeNode(1, TreeNode(2), TreeNode(3))
print("root.val =", root.val)
print("root.left.val =", root.left.val, "| root.right.val =", root.right.val)
print("root.left.left =", root.left.left)   # None: node 2 không có con trái

root.val = 1
root.left.val = 2 | root.right.val = 3
root.left.left = None


### 2.2. Dựng cây từ mảng kiểu LeetCode (level-order, có `null`)

LeetCode mô tả cây bằng mảng duyệt theo **từng tầng** (level order), dùng `null` cho vị trí trống. Ví dụ `[3,5,1,6,2,9,8,null,null,7,4]`.

Hàm `build_tree` dùng **hàng đợi** (`deque` của module `collections`): lấy node ra theo thứ tự, gắn lần lượt con trái rồi con phải từ mảng. Trong Python ta dùng `None` thay cho `null`.

`deque` giống list nhưng `popleft()` (lấy phần tử **đầu**) chạy `O(1)`, còn `list.pop(0)` là `O(n)`.

In [2]:
from collections import deque

def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:      # con trái
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:      # con phải
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def to_level_list(root: Optional[TreeNode]) -> list:
    # Ngược lại build_tree: in cây ra mảng level-order, bỏ None thừa ở cuối
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

t1 = build_tree([3, 5, 1, 6, 2, 9, 8, None, None, 7, 4])
print(to_level_list(t1))

[3, 5, 1, 6, 2, 9, 8, None, None, 7, 4]


### 2.3. Thế nào là "lá"?

**Lá** = node **không có con nào**: `node.left is None and node.right is None`.

Lỗi hay gặp: coi node chỉ thiếu **một** con là lá. Node `2` trong cây `[1,2,null,3]` có con trái `3` nên **không phải** lá.

In [3]:
def is_leaf(node: TreeNode) -> bool:
    return node.left is None and node.right is None

t = build_tree([1, 2, None, 3])     # 1 -> left 2 -> left 3
print("1 là lá?", is_leaf(t))
print("2 là lá?", is_leaf(t.left))
print("3 là lá?", is_leaf(t.left.left))

1 là lá? False
2 là lá? False
3 là lá? True


### 2.4. Đệ quy (recursion) trên cây

Cây có cấu trúc **tự đồng dạng**: cây con trái và cây con phải cũng là cây. Nên nhiều bài cây viết bằng **đệ quy**:

1. **Trường hợp cơ sở**: node là `None` → không làm gì.
2. **Bước đệ quy**: xử lý cây con trái, rồi cây con phải.

Ví dụ đếm số node: `count(node) = 1 + count(left) + count(right)`, với `count(None) = 0`.

In [4]:
def count_nodes(node: Optional[TreeNode]) -> int:
    if node is None:          # cơ sở
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

print("Số node của t1:", count_nodes(t1))

Số node của t1: 9


### 2.5. DFS: thứ tự "trái trước, phải sau" sinh ra lá theo trái → phải

**DFS** (Depth-First Search) đi sâu nhất có thể theo một nhánh rồi mới quay lại. Nếu ta luôn vào **con trái trước con phải**, thì các lá được gặp đúng theo thứ tự từ trái sang phải khi nhìn cây.

Dưới đây in thứ tự DFS **preorder** (thăm node, rồi trái, rồi phải) và đánh dấu node nào là lá.

In [5]:
def print_dfs(node: Optional[TreeNode], depth: int = 0) -> None:
    if node is None:
        return
    tag = "  <- LÁ" if is_leaf(node) else ""
    print("    " * depth + str(node.val) + tag)
    print_dfs(node.left, depth + 1)
    print_dfs(node.right, depth + 1)

print_dfs(t1)

3
    5
        6  <- LÁ
        2
            7  <- LÁ
            4  <- LÁ
    1
        9  <- LÁ
        8  <- LÁ


### 2.6. So sánh hai list bằng `==`

Trong Python, `list1 == list2` là `True` khi **cùng độ dài** và **từng phần tử ở cùng vị trí bằng nhau**. Thứ tự quan trọng: `[2, 3] != [3, 2]`.

Đây chính là cách "so sánh dãy lá" đơn giản nhất.

In [6]:
print([6, 7, 4, 9, 8] == [6, 7, 4, 9, 8])
print([2, 3] == [3, 2])
print([1, 2] == [1, 2, 3])

True
False
False


## 3. Cách 1 — Trực giác: gom toàn bộ lá vào list rồi so sánh

Câu hỏi: *"Hai cây leaf-similar nghĩa là gì?"* → "dãy lá của cây 1 == dãy lá của cây 2".

Vậy cứ làm đúng định nghĩa:
1. Viết hàm `collect_leaves(root)` trả về list các giá trị lá (DFS trái trước phải sau).
2. Trả về `collect_leaves(root1) == collect_leaves(root2)`.

Phần 1 tách riêng: hàm đệ quy có tham số `out` (list dùng chung), mỗi khi gặp lá thì `out.append(node.val)`.

In [7]:
def collect_leaves(root: Optional[TreeNode]) -> list[int]:
    out: list[int] = []

    def dfs(node: Optional[TreeNode]) -> None:
        if node is None:
            return
        if node.left is None and node.right is None:
            out.append(node.val)      # gặp lá -> ghi lại
            return
        dfs(node.left)                # trái trước
        dfs(node.right)               # phải sau

    dfs(root)
    return out

t2 = build_tree([3, 5, 1, 6, 7, 4, 2, None, None, None, None, None, None, 9, 8])
print("Lá cây 1:", collect_leaves(t1))
print("Lá cây 2:", collect_leaves(t2))

Lá cây 1: [6, 7, 4, 9, 8]
Lá cây 2: [6, 7, 4, 9, 8]


### 3.1. Hàm lồng nhau (nested function) và biến dùng chung

`dfs` được định nghĩa **bên trong** `collect_leaves`. Nó nhìn thấy được biến `out` của hàm bên ngoài (closure). Vì ta chỉ gọi `out.append(...)` (sửa nội dung list, không gán lại `out = ...`) nên không cần từ khóa `nonlocal`.

Ghép lại thành lời giải cách 1:

In [8]:
def leaf_similar_lists(root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
    return collect_leaves(root1) == collect_leaves(root2)

print(leaf_similar_lists(t1, t2))                                   # True
print(leaf_similar_lists(build_tree([1, 2, 3]), build_tree([1, 3, 2])))  # False

True
False


### 3.2. Đánh giá cách 1

Gọi `n1`, `n2` là số node của hai cây, `h1`, `h2` là chiều cao, `L1`, `L2` là số lá.

- Thời gian: `O(n1 + n2)` — mỗi node được thăm 1 lần.
- Bộ nhớ: `O(L1 + L2)` cho hai list lá + `O(h1 + h2)` cho ngăn xếp đệ quy.

Cách này **đã đủ tốt** cho LeetCode (cây tối đa 200 node). Nhưng có 2 điểm có thể cải thiện — đó là "nút thắt" ở mục sau.

## 4. Nút thắt: phải duyệt hết cả hai cây dù lá đầu tiên đã khác

Hình dung hai cây rất lớn, và **lá đầu tiên** đã khác nhau (`1` vs `2`). Cách 1 vẫn:
- duyệt toàn bộ cây 1,
- duyệt toàn bộ cây 2,
- mới so sánh.

Lãng phí! Lý tưởng là: lấy **lá thứ k** của cây 1 và **lá thứ k** của cây 2, so sánh ngay, khác thì **dừng**.

Ngoài ra, đệ quy sâu (cây suy biến thành "đường thẳng") có thể chạm giới hạn đệ quy của Python (~1000). Với đề này (≤200 node) chưa sao, nhưng nên biết cách viết **không đệ quy** bằng stack.

**Insight:** "Dãy lá" có thể được sinh ra **từng phần tử một** (lazily). Python có công cụ sinh lần lượt rất gọn: **generator**.

## 5. Cách 2 — DFS lặp bằng stack (không đệ quy)

**Stack** (ngăn xếp) = "vào sau ra trước". Dùng list: `append` để đẩy, `pop()` để lấy phần tử **cuối**.

Để vẫn gặp **trái trước**, khi xử lý một node ta đẩy **con phải trước, con trái sau** — vì con trái được đẩy sau nên sẽ được `pop` ra trước.

In [9]:
stack = []
stack.append("phải")
stack.append("trái")
print("pop lần 1:", stack.pop())   # trái
print("pop lần 2:", stack.pop())   # phải

pop lần 1: trái
pop lần 2: phải


In [10]:
def collect_leaves_iter(root: Optional[TreeNode]) -> list[int]:
    out: list[int] = []
    stack = [root] if root else []
    while stack:
        node = stack.pop()
        if node.left is None and node.right is None:
            out.append(node.val)
            continue
        if node.right:            # đẩy phải trước
            stack.append(node.right)
        if node.left:             # đẩy trái sau -> pop ra trước
            stack.append(node.left)
    return out

def leaf_similar_stack(root1, root2) -> bool:
    return collect_leaves_iter(root1) == collect_leaves_iter(root2)

print(collect_leaves_iter(t1))
print(leaf_similar_stack(t1, t2), leaf_similar_stack(build_tree([1, 2, 3]), build_tree([1, 3, 2])))

[6, 7, 4, 9, 8]
True False


## 6. Cách 3 — Generator: sinh từng lá và dừng sớm

### 6.1. `yield` là gì?

Hàm có `yield` là **generator function**. Gọi nó không chạy ngay mà trả về một **generator**; mỗi lần ta xin phần tử tiếp theo (bằng `next(...)` hoặc vòng `for`), hàm chạy tiếp đến `yield` kế tiếp rồi **tạm dừng**.

`yield from g` = "lần lượt yield mọi phần tử của generator `g`" — rất tiện cho đệ quy.

In [11]:
def count_up(n):
    for i in range(n):
        print("  (đang sinh", i, ")")
        yield i

g = count_up(3)
print("Đã tạo generator, chưa chạy gì.")
print("next ->", next(g))
print("next ->", next(g))
print("Phần còn lại:", list(g))

Đã tạo generator, chưa chạy gì.
  (đang sinh 0 )
next -> 0
  (đang sinh 1 )
next -> 1
  (đang sinh 2 )
Phần còn lại: [2]


In [12]:
def leaves_gen(node: Optional[TreeNode]):
    if node is None:
        return
    if node.left is None and node.right is None:
        yield node.val
        return
    yield from leaves_gen(node.left)
    yield from leaves_gen(node.right)

print(list(leaves_gen(t1)))

[6, 7, 4, 9, 8]


### 6.2. So sánh hai generator song song: `zip_longest`

`zip(a, b)` ghép cặp phần tử, nhưng **dừng ở dãy ngắn hơn** → nếu cây 1 có lá `[1,2]` còn cây 2 có `[1,2,3]`, `zip` sẽ không thấy lá `3` dư → kết luận sai!

`itertools.zip_longest(a, b, fillvalue=X)` ghép đến hết dãy dài hơn, chỗ thiếu điền `X`. Ta dùng một giá trị "canh gác" (sentinel) không thể là giá trị lá, ví dụ `None` (giá trị node trong `[0,200]` nên không trùng).

In [13]:
from itertools import zip_longest

print(list(zip([1, 2], [1, 2, 3])))
print(list(zip_longest([1, 2], [1, 2, 3], fillvalue=None)))

[(1, 1), (2, 2)]
[(1, 1), (2, 2), (None, 3)]


In [14]:
def leaf_similar_gen(root1, root2) -> bool:
    for a, b_ in zip_longest(leaves_gen(root1), leaves_gen(root2), fillvalue=None):
        if a != b_:
            return False          # dừng sớm ngay lá khác đầu tiên
    return True

print(leaf_similar_gen(t1, t2))
print(leaf_similar_gen(build_tree([1, 2, 3]), build_tree([1, 3, 2])))
print(leaf_similar_gen(build_tree([1, 2]), build_tree([5, 2, 3])))   # [2] vs [2,3]

True
False
False


## 7. Cách 4 — Hai stack chạy xen kẽ (dừng sớm, không đệ quy)

Kết hợp ý cách 2 và cách 3: mỗi cây có một stack riêng. Hàm `next_leaf(stack)` pop cho tới khi gặp lá thì trả về giá trị lá đó. Vòng chính: lấy lá kế tiếp của từng cây, so sánh, khác → `False`. Khi cả hai stack rỗng cùng lúc → `True`.

Đây là cách thường thấy trong phỏng vấn khi người ta hỏi "có thể không đệ quy và dừng sớm không?".

In [15]:
def next_leaf(stack: list) -> Optional[int]:
    while stack:
        node = stack.pop()
        if node.left is None and node.right is None:
            return node.val
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return None                     # hết lá

def leaf_similar_two_stacks(root1, root2) -> bool:
    s1, s2 = [root1], [root2]
    while s1 or s2:
        if next_leaf(s1) != next_leaf(s2):
            return False
    return True

print(leaf_similar_two_stacks(t1, t2))
print(leaf_similar_two_stacks(build_tree([1, 2, 3]), build_tree([1, 3, 2])))

True
False


Vì sao vòng `while s1 or s2` an toàn? Nếu một stack đã rỗng, `next_leaf` trả `None`; stack kia nếu còn lá sẽ trả một số ≠ `None` → `False`. Nếu stack kia còn node nhưng **không còn lá**… điều này không thể xảy ra: mọi cây con khác rỗng đều có ít nhất một lá. Nên khi cả hai đều trả `None` thì cả hai stack đều rỗng và vòng lặp kết thúc.

## 8. TRACE từng bước

### 8.1. Bảng trace cách 2 (stack) trên cây 1 `[3,5,1,6,2,9,8,null,null,7,4]`

| Bước | pop | Là lá? | Đẩy vào stack | stack sau (đỉnh bên phải) | out |
|---:|---:|:---:|---|---|---|
| 1 | 3 | không | 1, 5 | [1, 5] | [] |
| 2 | 5 | không | 2, 6 | [1, 2, 6] | [] |
| 3 | 6 | **có** | — | [1, 2] | [6] |
| 4 | 2 | không | 4, 7 | [1, 4, 7] | [6] |
| 5 | 7 | **có** | — | [1, 4] | [6,7] |
| 6 | 4 | **có** | — | [1] | [6,7,4] |
| 7 | 1 | không | 8, 9 | [8, 9] | [6,7,4] |
| 8 | 9 | **có** | — | [8] | [6,7,4,9] |
| 9 | 8 | **có** | — | [] | [6,7,4,9,8] |

Chú ý cột "Đẩy vào stack": phải trước, trái sau → trái nằm trên đỉnh.

In [16]:
def trace_collect_leaves_iter(root: Optional[TreeNode]) -> list[int]:
    out, stack, step = [], [root], 1
    while stack:
        node = stack.pop()
        if node.left is None and node.right is None:
            out.append(node.val)
            print(f"Bước {step}: pop {node.val:>2} | LÁ  -> out={out} | stack={[n.val for n in stack]}")
        else:
            pushed = []
            if node.right:
                stack.append(node.right); pushed.append(node.right.val)
            if node.left:
                stack.append(node.left); pushed.append(node.left.val)
            print(f"Bước {step}: pop {node.val:>2} | đẩy {pushed} | stack={[n.val for n in stack]}")
        step += 1
    return out

trace_collect_leaves_iter(t1)

Bước 1: pop  3 | đẩy [1, 5] | stack=[1, 5]
Bước 2: pop  5 | đẩy [2, 6] | stack=[1, 2, 6]
Bước 3: pop  6 | LÁ  -> out=[6] | stack=[1, 2]
Bước 4: pop  2 | đẩy [4, 7] | stack=[1, 4, 7]
Bước 5: pop  7 | LÁ  -> out=[6, 7] | stack=[1, 4]
Bước 6: pop  4 | LÁ  -> out=[6, 7, 4] | stack=[1]
Bước 7: pop  1 | đẩy [8, 9] | stack=[8, 9]
Bước 8: pop  9 | LÁ  -> out=[6, 7, 4, 9] | stack=[8]
Bước 9: pop  8 | LÁ  -> out=[6, 7, 4, 9, 8] | stack=[]


[6, 7, 4, 9, 8]

### 8.2. Trace cách dừng sớm trên ví dụ 2

`root1 = [1,2,3]`, `root2 = [1,3,2]`: lá thứ nhất là `2` và `3` → khác → dừng ngay, **không cần** xem lá thứ hai.

In [17]:
def trace_leaf_similar_two_stacks(root1, root2) -> bool:
    s1, s2, k = [root1], [root2], 1
    while s1 or s2:
        a, c = next_leaf(s1), next_leaf(s2)
        print(f"Lá thứ {k}: cây1={a}, cây2={c} ->", "bằng" if a == c else "KHÁC, dừng")
        if a != c:
            return False
        k += 1
    print("Cả hai hết lá cùng lúc -> True")
    return True

print(trace_leaf_similar_two_stacks(build_tree([1, 2, 3]), build_tree([1, 3, 2])))
print()
print(trace_leaf_similar_two_stacks(t1, t2))

Lá thứ 1: cây1=2, cây2=3 -> KHÁC, dừng
False

Lá thứ 1: cây1=6, cây2=6 -> bằng
Lá thứ 2: cây1=7, cây2=7 -> bằng
Lá thứ 3: cây1=4, cây2=4 -> bằng
Lá thứ 4: cây1=9, cây2=9 -> bằng
Lá thứ 5: cây1=8, cây2=8 -> bằng
Cả hai hết lá cùng lúc -> True
True


### 8.3. Trường hợp "một cây có nhiều lá hơn"

Cây 1 `[1,2]` có lá `[2]`; cây 2 `[5,2,3]` có lá `[2,3]`. Lá thứ 1 bằng nhau; lá thứ 2: cây 1 trả `None`, cây 2 trả `3` → khác → `False`. Đây là lý do không được dùng `zip` thường.

In [18]:
print(trace_leaf_similar_two_stacks(build_tree([1, 2]), build_tree([5, 2, 3])))

Lá thứ 1: cây1=2, cây2=2 -> bằng
Lá thứ 2: cây1=None, cây2=3 -> KHÁC, dừng
False


## 9. Code nộp LeetCode

Bản ngắn gọn, dễ đọc nhất (cách 1). LeetCode có sẵn `TreeNode`; kiểu `TreeNode | None` hợp lệ ở Python 3.10+.

In [19]:
class Solution:
    def leafSimilar(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        def leaves(node, out):
            if node is None:
                return out
            if node.left is None and node.right is None:
                out.append(node.val)
            else:
                leaves(node.left, out)
                leaves(node.right, out)
            return out

        return leaves(root1, []) == leaves(root2, [])

sol = Solution()
print(sol.leafSimilar(t1, t2))
print(sol.leafSimilar(build_tree([1, 2, 3]), build_tree([1, 3, 2])))

True
False


### 9.1. Bản nộp dừng sớm (generator)

In [20]:
from itertools import zip_longest

class SolutionGen:
    def leafSimilar(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> bool:
        def gen(node):
            if node is None:
                return
            if node.left is None and node.right is None:
                yield node.val
                return
            yield from gen(node.left)
            yield from gen(node.right)

        sentinel = object()           # chắc chắn khác mọi giá trị lá
        return all(a == b for a, b in zip_longest(gen(root1), gen(root2), fillvalue=sentinel))

print(SolutionGen().leafSimilar(t1, t2), SolutionGen().leafSimilar(build_tree([1, 2]), build_tree([5, 2, 3])))

True False


`all(...)` trả `True` nếu mọi phần tử đúng, và **dừng ngay** khi gặp phần tử sai đầu tiên (short-circuit) — nên vẫn giữ được tính dừng sớm. `object()` tạo một đối tượng mới duy nhất, không bằng bất kỳ số nào → sentinel an toàn tuyệt đối.

## 10. Vì sao thuật toán đúng?

**Mệnh đề:** DFS luôn đi con trái trước con phải sẽ gặp các lá theo đúng thứ tự trái → phải.

**Lý do (quy nạp theo cây):**
- Cây chỉ có 1 node: node đó là lá duy nhất → đúng.
- Cây có gốc `r`: mọi lá của cây con trái nằm **bên trái** mọi lá của cây con phải (khi vẽ cây). DFS xử lý **xong toàn bộ** cây con trái (theo giả thiết quy nạp: đúng thứ tự) rồi mới sang cây con phải (cũng đúng thứ tự) → nối lại vẫn đúng thứ tự.

**Invariant của cách stack:** tại mọi thời điểm, các node trong stack, đọc từ **đỉnh xuống đáy**, là các cây con chưa thăm theo thứ tự trái → phải; `out` chứa đúng các lá nằm bên trái tất cả cây con trong stack.

**So sánh:** hai cây leaf-similar ⇔ hai dãy bằng nhau ⇔ cùng độ dài và từng vị trí bằng nhau — chính xác điều `==` (hoặc `zip_longest` + so sánh từng cặp) kiểm tra.

## 11. Phân tích độ phức tạp và so sánh các cách

Ký hiệu: `n = n1 + n2` (tổng số node), `h` = chiều cao lớn nhất, `L` = số lá.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Dừng sớm? | Đệ quy? |
|---|---|---|---|:---:|:---:|
| 1. List + đệ quy | gom lá rồi `==` | `O(n)` | `O(L + h)` | không | có |
| 2. List + stack | DFS lặp | `O(n)` | `O(L + h)` | không | không |
| 3. Generator | `yield` + `zip_longest` | `O(n)` worst, có thể ít hơn | `O(h)` | **có** | có |
| 4. Hai stack | `next_leaf` xen kẽ | `O(n)` worst, có thể ít hơn | `O(h)` | **có** | không |

- Worst case (cây giống nhau): mọi cách đều phải thăm hết node.
- Stack của DFS lặp chứa tối đa khoảng `O(h)` node (cộng các anh em phải đang chờ, vẫn bị chặn bởi `O(h)` với cây nhị phân).
- Với ràng buộc ≤ 200 node, cách 1 hoàn toàn đủ; cách 3/4 là kiến thức nâng cao đáng học.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu các cách

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

methods = {
    "lists": leaf_similar_lists,
    "stack": leaf_similar_stack,
    "gen": leaf_similar_gen,
    "two_stacks": leaf_similar_two_stacks,
    "Solution": Solution().leafSimilar,
    "SolutionGen": SolutionGen().leafSimilar,
}

cases = [
    ("đề VD1", [3,5,1,6,2,9,8,None,None,7,4], [3,5,1,6,7,4,2,None,None,None,None,None,None,9,8], True),
    ("đề VD2", [1,2,3], [1,3,2], False),
    ("1 node bằng", [1], [1], True),
    ("1 node khác", [1], [2], False),
    ("gốc khác, lá giống", [7], [9, 7], True),               # cây 2: 9 -> left 7
    ("số lá khác", [1,2], [5,2,3], False),
    ("lá trùng giá trị", [1,2,2], [3,2,None,2], False),       # [2,2] vs [2]
    ("cùng tập khác thứ tự", [0,1,2,3,4], [0,1,2,4,3], False),
    ("cây lệch trái vs lệch phải", [1,2,None,3], [5,None,4,None,3], True),
]
for name, a, c, exp in cases:
    for mname, f in methods.items():
        assert f(build_tree(a), build_tree(c)) == exp, (name, mname)
    check(name, leaf_similar_lists(build_tree(a), build_tree(c)), exp)
print("Tất cả", len(methods), "cách đều PASS mọi test.")

PASS  đề VD1: True
PASS  đề VD2: False
PASS  1 node bằng: True
PASS  1 node khác: False
PASS  gốc khác, lá giống: True
PASS  số lá khác: False
PASS  lá trùng giá trị: False
PASS  cùng tập khác thứ tự: False
PASS  cây lệch trái vs lệch phải: True
Tất cả 6 cách đều PASS mọi test.


### 12.1. Kiểm thử ngẫu nhiên có seed: các cách phải luôn đồng ý

Sinh cây ngẫu nhiên nhỏ, so tất cả cách với cách 1 (làm "oracle"). Dùng `random.seed(...)` để kết quả lặp lại được.

In [22]:
import random

def random_tree(n_nodes: int, max_val: int = 3) -> Optional[TreeNode]:
    if n_nodes == 0:
        return None
    left_size = random.randint(0, n_nodes - 1)
    return TreeNode(random.randint(0, max_val),
                    random_tree(left_size, max_val),
                    random_tree(n_nodes - 1 - left_size, max_val))

random.seed(872)
agree = 0
for _ in range(500):
    r1 = random_tree(random.randint(1, 8))
    r2 = random_tree(random.randint(1, 8))
    expected = leaf_similar_lists(r1, r2)
    for f in methods.values():
        assert f(r1, r2) == expected
    agree += 1
print("500 cặp cây ngẫu nhiên: mọi cách đồng ý =", agree == 500)

500 cặp cây ngẫu nhiên: mọi cách đồng ý = True


## 13. Lỗi người mới hay gặp

### Lỗi 1: Coi node "thiếu một con" là lá
```python
if node.left is None or node.right is None:   # SAI: dùng or
    out.append(node.val)
```
Node chỉ có con trái sẽ bị tính là lá, và lá thật bên dưới cũng bị tính → dãy sai. **Sửa:** dùng `and`.

In [23]:
def collect_leaves_wrong(node, out):
    if node is None:
        return out
    if node.left is None or node.right is None:    # SAI
        out.append(node.val)
    collect_leaves_wrong(node.left, out)
    collect_leaves_wrong(node.right, out)
    return out

t = build_tree([1, 2, None, 3])
print("Sai :", collect_leaves_wrong(t, []))   # [1, 2, 3] - node 1 và 2 bị coi là lá
print("Đúng:", collect_leaves(t))             # [3]

Sai : [1, 2, 3]
Đúng: [3]


### Lỗi 2: So sánh bằng `set` hoặc `sorted`
```python
return set(leaves1) == set(leaves2)      # SAI
```
Bỏ mất **thứ tự** và **số lần lặp**. Ví dụ 2 của đề (`[2,3]` vs `[3,2]`) sẽ ra `True` sai.

### Lỗi 3: Dùng `zip` thường khi so sánh dừng sớm
`zip` cắt theo dãy ngắn hơn → `[2]` vs `[2,3]` bị coi là bằng.

In [24]:
l1, l2 = [2, 3], [3, 2]
print("set so sánh (sai):", set(l1) == set(l2), "| list so sánh (đúng):", l1 == l2)

def wrong_zip(root1, root2):
    return all(a == c for a, c in zip(leaves_gen(root1), leaves_gen(root2)))

print("zip thường (sai):", wrong_zip(build_tree([1, 2]), build_tree([5, 2, 3])))
print("zip_longest (đúng):", leaf_similar_gen(build_tree([1, 2]), build_tree([5, 2, 3])))

set so sánh (sai): True | list so sánh (đúng): False
zip thường (sai): True
zip_longest (đúng): False


### Lỗi 4: Đẩy con trái vào stack trước
Với stack, phần tử vào **sau** ra **trước**. Đẩy trái trước → phải được xử lý trước → dãy lá bị đảo thành phải → trái.

### Lỗi 5: Dùng list mặc định làm tham số `def f(node, out=[])`
Giá trị mặc định chỉ được tạo **một lần** khi định nghĩa hàm → các lần gọi sau dùng chung list cũ, lá tích lũy sai. Luôn truyền `[]` mới hoặc dùng `out=None` rồi tạo bên trong.

In [25]:
def bad_default(node, out=[]):           # SAI: list mặc định dùng chung
    if node is None:
        return out
    if node.left is None and node.right is None:
        out.append(node.val)
    else:
        bad_default(node.left, out); bad_default(node.right, out)
    return out

print(bad_default(build_tree([1, 2, 3])))
print(bad_default(build_tree([1, 2, 3])), "<- lần 2 bị cộng dồn!")

[2, 3]
[2, 3, 2, 3] <- lần 2 bị cộng dồn!


## 14. Mẹo nhận dạng pattern

- Đề nói "các lá **từ trái sang phải**", "thứ tự in-order", "đường đi từ gốc" → nghĩ ngay tới **DFS**, con trái trước.
- "So sánh hai cấu trúc theo **một dãy rút ra** từ chúng" → biến mỗi cấu trúc thành **dãy** (list hoặc generator) rồi so sánh dãy.
- Cần **dừng sớm** khi so sánh hai dãy dài → generator + `zip_longest` hoặc hai iterator chạy song song.
- Muốn tránh đệ quy → **stack thủ công**, nhớ đẩy theo thứ tự ngược với thứ tự muốn thăm.

Bài cùng họ: 100 Same Tree, 101 Symmetric Tree, 94 Binary Tree Inorder Traversal, 173 BST Iterator, 1305 All Elements in Two BSTs.

## 15. Biến thể

### 15.1. Trả về dãy lá thay vì True/False
Đã có: `collect_leaves`.

### 15.2. Đếm số lá ở mỗi cây và tổng giá trị lá
### 15.3. "Leaf-similar bỏ qua thứ tự" — so sánh **multiset** lá bằng `Counter`
`collections.Counter` đếm số lần xuất hiện mỗi giá trị; hai `Counter` bằng nhau khi mọi giá trị có cùng số lần.

In [26]:
from collections import Counter

def leaf_multiset_equal(root1, root2) -> bool:
    return Counter(leaves_gen(root1)) == Counter(leaves_gen(root2))

print("Bỏ qua thứ tự [2,3] vs [3,2]:", leaf_multiset_equal(build_tree([1, 2, 3]), build_tree([1, 3, 2])))
print("Bỏ qua thứ tự [2,2] vs [2]  :", leaf_multiset_equal(build_tree([1, 2, 2]), build_tree([3, 2])))

def leaf_count_and_sum(root):
    vals = collect_leaves(root)
    return len(vals), sum(vals)

print("t1: (số lá, tổng lá) =", leaf_count_and_sum(t1))

Bỏ qua thứ tự [2,3] vs [3,2]: True
Bỏ qua thứ tự [2,2] vs [2]  : False
t1: (số lá, tổng lá) = (5, 34)


### 15.4. Lá theo thứ tự phải → trái
Chỉ cần đổi thứ tự đệ quy (phải trước) — hoặc đơn giản đảo list. Biến thể này giúp kiểm tra bạn có hiểu "thứ tự thăm quyết định thứ tự dãy" không.

In [27]:
def leaves_right_to_left(node):
    if node is None:
        return
    if node.left is None and node.right is None:
        yield node.val
        return
    yield from leaves_right_to_left(node.right)
    yield from leaves_right_to_left(node.left)

print(list(leaves_right_to_left(t1)), "==", collect_leaves(t1)[::-1])

[8, 9, 4, 7, 6] == [8, 9, 4, 7, 6]


## 16. Bài tập tự luyện (từ dễ đến khó)

Các hàm dưới đây **chỉ định nghĩa**, không được gọi khi chạy notebook. Tự làm xong rồi bỏ comment các dòng test.

### Bài 1 — Tính tay
Cho cây `[4, 2, 6, 1, 3, 5, 7]`. Viết ra dãy lá trái → phải. Cây `[4, 2, 6, None, 3, 5]` thì sao? Hai cây có leaf-similar không?

### Bài 2 — Đếm lá
Viết `count_leaves(root)` trả về số lá của cây.

In [28]:
def count_leaves(root) -> int:
    # TODO: base case None -> 0; lá -> 1; còn lại -> tổng hai con
    pass

# assert count_leaves(build_tree([3,5,1,6,2,9,8,None,None,7,4])) == 5
# assert count_leaves(build_tree([1])) == 1

### Bài 3 — Điền chỗ trống: DFS lặp lấy lá

In [29]:
def leaves_iter_exercise(root) -> list[int]:
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        # TODO 1: nếu node là lá -> out.append, continue
        # TODO 2: đẩy con phải (nếu có)
        # TODO 3: đẩy con trái (nếu có)
        pass
    return out

# assert leaves_iter_exercise(build_tree([3,5,1,6,2,9,8,None,None,7,4])) == [6,7,4,9,8]

### Bài 4 — Tự viết lại `leafSimilar` bằng generator + dừng sớm

In [30]:
def leaf_similar_exercise(root1, root2) -> bool:
    # TODO: viết generator lá, so sánh bằng zip_longest với sentinel
    pass

# assert leaf_similar_exercise(build_tree([1,2,3]), build_tree([1,3,2])) is False
# assert leaf_similar_exercise(build_tree([1,2]), build_tree([5,2,3])) is False

### Bài 5 — Biến thể: độ sâu của các lá
Trả về list các cặp `(giá trị lá, độ sâu)` theo thứ tự trái → phải, gốc có độ sâu 0.

In [31]:
def leaves_with_depth(root) -> list[tuple[int, int]]:
    # TODO: DFS mang theo depth
    pass

# assert leaves_with_depth(build_tree([1,2,3,None,4])) == [(4, 2), (3, 1)]

### Bài 6 — Mini challenge: chỉ số lá khác nhau đầu tiên
Trả về chỉ số `k` (từ 0) của vị trí đầu tiên mà dãy lá hai cây khác nhau; nếu giống hệt trả `-1`. Yêu cầu: dừng sớm, không gom toàn bộ lá vào list.

In [32]:
def first_diff_leaf(root1, root2) -> int:
    # TODO: zip_longest + enumerate
    pass

# assert first_diff_leaf(build_tree([1,2,3]), build_tree([1,3,2])) == 0
# assert first_diff_leaf(build_tree([1,2]), build_tree([5,2,3])) == 1
# assert first_diff_leaf(build_tree([1,2,3]), build_tree([9,2,3])) == -1

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Lá = node không con | Vẽ cây theo tầng | `[1,3,5,7]`; cây 2: `[3,5]` → không |
| 2 | Đệ quy 3 trường hợp | `if node is None` / `if lá` | `return count(l) + count(r)` |
| 3 | Giống cách 2 | phải trước, trái sau | `if node.right: stack.append(node.right)` |
| 4 | Generator + so sánh song song | `yield from` | `for a, c in zip_longest(..., fillvalue=sentinel)` |
| 5 | Mang thêm tham số | `dfs(node, depth)` | gọi con với `depth + 1` |
| 6 | Đánh số cặp lá | `enumerate(zip_longest(...))` | `if a != c: return k` |

## 18. Đáp án tham khảo

Chỉ xem sau khi đã tự làm.

**Bài 1:** cây `[4,2,6,1,3,5,7]` có lá `[1,3,5,7]`. Cây `[4,2,6,None,3,5]`: node 2 chỉ có con phải 3, node 6 chỉ có con trái 5 → lá `[3,5]`. Hai dãy khác → không leaf-similar.

In [33]:
def ans_count_leaves(root) -> int:
    if root is None:
        return 0
    if root.left is None and root.right is None:
        return 1
    return ans_count_leaves(root.left) + ans_count_leaves(root.right)

def ans_leaves_iter(root) -> list[int]:
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        if node.left is None and node.right is None:
            out.append(node.val)
            continue
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return out

def ans_leaf_similar(root1, root2) -> bool:
    def gen(node):
        if node is None:
            return
        if node.left is None and node.right is None:
            yield node.val
            return
        yield from gen(node.left)
        yield from gen(node.right)
    sentinel = object()
    for a, c in zip_longest(gen(root1), gen(root2), fillvalue=sentinel):
        if a != c:
            return False
    return True

def ans_leaves_with_depth(root) -> list[tuple[int, int]]:
    out = []
    def dfs(node, depth):
        if node is None:
            return
        if node.left is None and node.right is None:
            out.append((node.val, depth))
            return
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return out

def ans_first_diff_leaf(root1, root2) -> int:
    sentinel = object()
    for k, (a, c) in enumerate(zip_longest(leaves_gen(root1), leaves_gen(root2), fillvalue=sentinel)):
        if a != c:
            return k
    return -1

assert collect_leaves(build_tree([4,2,6,1,3,5,7])) == [1,3,5,7]
assert collect_leaves(build_tree([4,2,6,None,3,5])) == [3,5]
assert ans_count_leaves(build_tree([3,5,1,6,2,9,8,None,None,7,4])) == 5
assert ans_count_leaves(build_tree([1])) == 1
assert ans_leaves_iter(build_tree([3,5,1,6,2,9,8,None,None,7,4])) == [6,7,4,9,8]
assert ans_leaf_similar(build_tree([1,2,3]), build_tree([1,3,2])) is False
assert ans_leaf_similar(build_tree([1,2]), build_tree([5,2,3])) is False
assert ans_leaf_similar(t1, t2) is True
assert ans_leaves_with_depth(build_tree([1,2,3,None,4])) == [(4, 2), (3, 1)]
assert ans_first_diff_leaf(build_tree([1,2,3]), build_tree([1,3,2])) == 0
assert ans_first_diff_leaf(build_tree([1,2]), build_tree([5,2,3])) == 1
assert ans_first_diff_leaf(build_tree([1,2,3]), build_tree([9,2,3])) == -1
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh và checklist submit

```python
class Solution:
    def leafSimilar(self, root1, root2) -> bool:
        def leaves(node, out):
            if node is None:
                return out
            if node.left is None and node.right is None:   # lá: KHÔNG có con nào
                out.append(node.val)
            else:
                leaves(node.left, out)                      # trái trước
                leaves(node.right, out)                     # phải sau
            return out
        return leaves(root1, []) == leaves(root2, [])       # so sánh có thứ tự
```

**Công thức tư duy:** *"Rút mỗi cây thành một dãy bằng DFS trái→phải, rồi so sánh dãy."*

**Checklist trước khi nộp:**
- [ ] Điều kiện lá dùng `and` (cả hai con đều `None`)?
- [ ] Duyệt con trái **trước** con phải (đệ quy) / đẩy phải trước trái (stack)?
- [ ] So sánh **list** (có thứ tự, có trùng lặp), không dùng `set`/`sorted`?
- [ ] Nếu dừng sớm: dùng `zip_longest` + sentinel, không dùng `zip`?
- [ ] Không dùng list mặc định `out=[]` trong chữ ký hàm?
- [ ] Đã test ví dụ đề, cây 1 node, cây khác số lá?